# Tutorial 02: Model Training with Config

In this tutorial, we will use the **YAML configuration** to define hyperparameters and train our model using the processed dataset. The architecture is selected via the Model Factory, allowing you to switch between models with a single line change.

## 1. Setup and Imports

In [1]:
import os
import sys
import yaml
import pickle

# Add src to path
sys.path.append(os.path.abspath('../../src'))

from bioacoustica.training.trainer import Trainer
print("Trainer module loaded.")

2026-02-24 20:02:01.997485: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-24 20:02:02.025618: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 20:02:02.481258: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Trainer module loaded.


## 2. Load Config and Data

In [2]:
CONFIG_PATH = "../../configs/gibbon.yaml"
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

DATA_DIR = "../../data/processed"

print(f"Loading processed data from {DATA_DIR}...")
with open(os.path.join(DATA_DIR, "X.pkl"), "rb") as f:
    X = pickle.load(f)

with open(os.path.join(DATA_DIR, "Y.pkl"), "rb") as f:
    Y = pickle.load(f)

print(f"Data loaded. Input shape: {X.shape}, Samples: {len(X)}")

Loading processed data from ../../data/processed...
Data loaded. Input shape: (3356, 128, 126), Samples: 3356


## 3. Select Architecture & Train Model

The **Model Factory** (`get_model`) allows to switch between architectures:

| `architecture` | Best For | Notes |
| --- | --- | --- |
| `custom_cnn` | Low resources, fast training | Grayscale 1-channel spectrograms |
| `mobilenet_v2` | Edge deployment | Auto-converts 1-ch to RGB |
| `efficientnet_b0` | Best accuracy/efficiency | Auto-converts 1-ch to RGB |

> **Change `architecture` in `configs/gibbon.yaml`** to switch models. Set `fine_tune: true` to unfreeze the pre-trained backbone.

In [3]:
trainer = Trainer(output_dir="../../models", seed=config['training']['seed'])

arch = config['training']['architecture']
print(f"Starting training with architecture: '{arch}' for {config['training']['epochs']} epochs...")

model_path = trainer.train(
    X=X,
    Y=Y,
    class_order=config['classes']['order'],
    model_architecture=arch,
    epochs=config['training']['epochs'],
    batch_size=config['training']['batch_size'],
    dropout_rate=config['training'].get('dropout_rate', 0.3),
    fine_tune=config['training'].get('fine_tune', False)
)

print(f"\nTraining complete. Model saved at: {model_path}")

Starting training with architecture: 'custom_cnn' for 20 epochs...
Original split: Train=(2684, 128, 126), Val=(672, 128, 126)
Augmenting 'no-gibbon' by 273 samples...
Augmenting 'gibbon' by 50 samples...
Final shapes: X_train=(3007, 128, 126), y_train=(3007, 2)


2026-02-24 20:02:08.478307: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-24 20:02:08.528248: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Epoch 1/20
94/94 [==============================] - ETA: 0s - loss: 0.9881 - accuracy: 0.7336
Epoch 1: val_loss improved from inf to 0.33864, saving model to ../../models/custom_cnn_20260224_200208_best.h5
94/94 [==============================] - 25s 259ms/step - loss: 0.9881 - accuracy: 0.7336 - val_loss: 0.3386 - val_accuracy: 0.9033 - lr: 0.0010
Epoch 2/20


/home/milanto/Documents/Antigravity/bioacoustic-detection-pipeline/venv/lib/python3.9/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


94/94 [==============================] - ETA: 0s - loss: 0.1856 - accuracy: 0.9701
Epoch 2: val_loss improved from 0.33864 to 0.14970, saving model to ../../models/custom_cnn_20260224_200208_best.h5
94/94 [==============================] - 24s 257ms/step - loss: 0.1856 - accuracy: 0.9701 - val_loss: 0.1497 - val_accuracy: 0.9658 - lr: 0.0010
Epoch 3/20
94/94 [==============================] - ETA: 0s - loss: 0.1259 - accuracy: 0.9734
Epoch 3: val_loss improved from 0.14970 to 0.11183, saving model to ../../models/custom_cnn_20260224_200208_best.h5
94/94 [==============================] - 24s 257ms/step - loss: 0.1259 - accuracy: 0.9734 - val_loss: 0.1118 - val_accuracy: 0.9717 - lr: 0.0010
Epoch 4/20
94/94 [==============================] - ETA: 0s - loss: 0.1212 - accuracy: 0.9777
Epoch 4: val_loss improved from 0.11183 to 0.08825, saving model to ../../models/custom_cnn_20260224_200208_best.h5
94/94 [==============================] - 24s 254ms/step - loss: 0.1212 - accuracy: 0.9777 -